# Lecture 5 · Probabilistic Models and Learning — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the material of **Lecture 5**: Bayesian networks (DAGs, the local Markov property, the factorisation
$P(X_1,\dots,X_n) = \prod_iP(X_i\mid pa(X_i))$), inference by restriction, combination and marginalisation, **maximum likelihood** learning
(counting for discrete CPTs, closed forms for the Gaussian), missing-data mechanisms (MCAR, MAR, MNAR) and the **EM algorithm**, the zero-probability
problem and **Laplace smoothing**, and fully Bayesian learning with the **Beta-Bernoulli** model and its posterior predictive. Code exercises
mirror `exercises_02.ipynb` (Gaussian MLE, Naive Bayes from frequency tables) and implement inference and EM from scratch.

**20 exercises** · 🧠 Concept ×5 · ✍️ By hand ×8 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import itertools
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, beta
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.naive_bayes import CategoricalNB
from sklearn.mixture import GaussianMixture
rng = np.random.default_rng(5)

# A small Bayesian network used in several exercises:  Rain -> Wet <- Sprinkler   (all binary, index 1 = true)
P_R = np.array([0.8, 0.2])                    # P(R)
P_S = np.array([0.6, 0.4])                    # P(S)
P_W1_given_RS = np.array([[0.05, 0.80],       # P(W=1 | R=0, S=0), P(W=1 | R=0, S=1)
                          [0.90, 0.99]])      # P(W=1 | R=1, S=0), P(W=1 | R=1, S=1)

# Data from exercises_02.ipynb, Part 1
data_gauss = np.array([4.04078064, -0.55665445, 3.05080334, 5.08400317, 2.72089663, 0.71922335])

# A -> B network with missing values of A (np.nan = missing)
A_obs = np.array([1, 1, 1, 1, 0, 0, 0, np.nan, np.nan, np.nan])
B_obs = np.array([1, 1, 1, 0, 0, 0, 1, 1, 1, 0])

## Part A · Concepts

### Exercise 1 · MCAR, MAR or non-ignorable?
*🧠 Concept · ★☆☆*

Classify each missingness mechanism and say whether EM on the observed data gives valid estimates without modelling the mechanism.

(a) A lab robot randomly drops 5 % of measurements because of a software bug.
(b) Older patients (age is always recorded) are less likely to have a cholesterol test.
(c) People with very high income skip the income question.
(d) An exit poll where supporters of one party systematically refuse to answer the vote question.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: does the probability of being missing depend on nothing, on observed variables only, or on the missing value itself?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **MCAR**: independent of everything. (b) **MAR**: depends only on the observed age. (c) **Non-ignorable (MNAR)**: depends on the unobserved income.
(d) **Non-ignorable**: missingness depends on the vote itself.

Under MCAR and MAR the missingness mechanism can be ignored for likelihood-based learning (EM on the observed data is fine). Under MNAR it must be modelled explicitly, otherwise estimates are biased
(e.g. average income underestimated).

</details>

### Exercise 2 · What a missing edge says
*🧠 Concept · ★☆☆*

For the network Rain → Wet ← Sprinkler:
(a) Write the factorisation of $P(R, S, W)$. (b) What independence does the **missing** edge between Rain and Sprinkler encode (local Markov property)?
(c) Are $R$ and $S$ still independent once you observe $W = 1$? Give the intuition (Exercise 9 computes it).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$nd(S)$ contains $R$, and $pa(S) = \emptyset$. $W$ is a collider (v-structure).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $P(R,S,W) = P(R)\,P(S)\,P(W\mid R,S)$.
(b) $S\perp\!\!\!\perp nd(S)\mid pa(S)$ with $pa(S) = \emptyset$: $S$ and $R$ are **marginally independent**.
(c) No. Observing the common effect couples the causes: if the grass is wet and we learn the sprinkler was on, rain becomes less likely (**explaining away**).
Conditioning on a collider *creates* dependence.

</details>

### Exercise 3 · The zero-count trap
*🧠 Concept · ★★☆*

In `exercises_02.ipynb` the class `draw` never co-occurs with `MH = 1a` in the 20 training rows.
(a) What posterior does the MLE Naive Bayes assign to `draw` for a hand with `MH = 1a`, regardless of the other features, and why?
(b) How does Laplace smoothing fix it, and how is it related to a Beta/Dirichlet prior?
(c) Place this diagnosis and fix in **Box's loop**.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The score is a product. What is $0\cdot x$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\hat P(\text{MH}=1a\mid\text{draw}) = 0/N_{\text{draw}} = 0$, so the product $P(\text{draw})\prod_iP(\cdot\mid\text{draw}) = 0$ and the posterior of `draw` is exactly 0: one unseen
combination vetoes all other evidence. That is severe overfitting to a small sample.
(b) Add a pseudo-count $\alpha$ to every cell: $\frac{N_{jk}+\alpha}{\sum_{k'}(N_{jk'}+\alpha)}$. This is the posterior mean under a symmetric Dirichlet($\alpha$) prior (Beta($\alpha,\alpha$) in the binary case),
see Exercise 10.
(c) **Criticise** (detect impossible/zero predictions on test data), then **revise** (add smoothing or an informative prior) and go round the loop again: build, infer, criticise, revise.

</details>

### Exercise 4 · EM: guaranteed, but only locally
*🧠 Concept · ★★☆*

(a) What exactly does EM guarantee at every iteration? (b) Why can it still end in a poor solution, and what is the standard remedy?
(c) Why does complete data not need EM at all?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

EM is a coordinate-ascent method on a lower bound of the marginal log-likelihood.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The observed-data (marginal) log-likelihood never decreases: $\ell(\theta^{t+1})\ge\ell(\theta^t)$ (Exercises 18 and 20 check it).
(b) With hidden variables $\ell(\theta)$ is non-concave with several local maxima/saddles (e.g. label switching in mixtures). EM converges to a stationary point that depends on $\theta^0$.
Remedy: **random restarts** and keep the run with the highest likelihood (or smarter initialisation such as k-means for GMMs).
(c) With complete data the log-likelihood decomposes into independent local terms per CPT whose maximisers are closed-form counts, one step, no iteration.

</details>

### Exercise 5 · MLE, MAP and the posterior predictive
*🧠 Concept · ★★☆*

For a coin with $\text{Beta}(a,b)$ prior and $N_1$ heads, $N_0$ tails, write the three point predictions for $P(X_{N+1} = 1)$:
MLE, MAP (posterior mode) and the Bayesian posterior predictive. Which ones can return exactly 0 after 3 tails and no heads? What happens to all three as $N\to\infty$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The mode of $\text{Beta}(\alpha,\beta)$ is $\frac{\alpha-1}{\alpha+\beta-2}$ (for $\alpha,\beta>1$), the mean is $\frac{\alpha}{\alpha+\beta}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MLE $\frac{N_1}{N_1+N_0}$; MAP $\frac{a+N_1-1}{a+b+N-2}$; predictive $\frac{a+N_1}{a+b+N}$.
After 3 tails: the MLE gives 0; the MAP gives 0 if $a = 1$ (uniform prior) and is positive for $a>1$; the predictive is $\frac{a}{a+b+3}>0$ for any $a>0$.
As $N\to\infty$ the prior's pseudo-counts become negligible and all three converge to the true frequency.

</details>

## Part B · By hand

### Exercise 6 · Gaussian MLE for the self-study data
*✍️ By hand · ★☆☆*

For the six values in `data_gauss` (from `exercises_02.ipynb`), compute $\hat\mu = \frac1N\sum x_i$ and $\hat\sigma = \sqrt{\frac1N\sum(x_i-\hat\mu)^2}$
(a calculator is fine). Why is $\hat\sigma^2$ biased, and what is the unbiased alternative?

In [ ]:
mu_hat = None
sigma_hat = None

_m, _s = norm.fit(data_gauss)
check('mu_hat', mu_hat, _m, tol=1e-3)
check('sigma_hat', sigma_hat, _s, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\sum x_i\approx15.0591$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\hat\mu\approx15.0591/6 = \mathbf{2.5098}$, $\hat\sigma\approx\mathbf{1.9112}$ (exactly what `norm.fit` returns for these six values).
*Note:* the lecture note quotes $\hat\mu = 2.1764$, $\hat\sigma = 2.0836$; those numbers do not follow from the six values printed there (presumably they come from the full
self-study array), so trust the computation.

$E[\hat\sigma^2] = \frac{N-1}{N}\sigma^2$ because $\hat\mu$ is fitted to the same data; the unbiased estimator divides by $N-1$ (`np.std(ddof=1)`), giving about 2.0937 here.

```python
mu_hat = data_gauss.mean()
sigma_hat = data_gauss.std()
```

</details>

### Exercise 7 · Deriving the Gaussian MLE
*✍️ By hand · ★★☆*

Starting from $\ln L(\mu,\sigma^2) = -\frac{1}{2\sigma^2}\sum_i(x_i-\mu)^2 - \frac N2\ln(2\pi\sigma^2)$, derive $\hat\mu$ and $\hat\sigma^2$ by setting the partial derivatives to zero.
Check that the stationary point in $\mu$ is a maximum.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Treat $v = \sigma^2$ as the variable: $\partial/\partial v$ of $-\frac{S}{2v} - \frac N2\ln v$ with $S = \sum(x_i-\mu)^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{\partial\ln L}{\partial\mu} = \frac{1}{\sigma^2}\sum_i(x_i-\mu) = 0\Rightarrow\hat\mu = \frac1N\sum_ix_i$; the second derivative $-N/\sigma^2<0$, so it is a maximum.

$\frac{\partial\ln L}{\partial v} = \frac{S}{2v^2} - \frac{N}{2v} = 0\Rightarrow\hat v = \frac SN = \frac1N\sum_i(x_i-\hat\mu)^2$.

</details>

### Exercise 8 · Why factorisation pays off: counting parameters
*✍️ By hand · ★☆☆*

Five binary variables $A,B,C,D,E$. (a) How many free parameters does an unrestricted joint table need? (b) How many does the Bayesian network
$A\to C\leftarrow B$, $C\to D$, $C\to E$ need? (c) For $n$ binary variables where each has at most $k$ parents, give the BN bound.

In [ ]:
n_full = None
n_bn = None

check('full joint', n_full, 2 ** 5 - 1)
_parents = {'A': 0, 'B': 0, 'C': 2, 'D': 1, 'E': 1}
check('Bayesian network', n_bn, sum(2 ** k for k in _parents.values()))

<details>
<summary><b>💡 Hint</b></summary>

A binary node with $k$ binary parents needs one number per parent configuration: $2^k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $2^5 - 1 = \mathbf{31}$. (b) $1 + 1 + 4 + 2 + 2 = \mathbf{10}$. (c) At most $n\,2^k$, linear in $n$ instead of exponential: this is what makes learning (and storing) the joint feasible.

```python
n_full = 31
n_bn = 10
```

</details>

### Exercise 9 · Inference by enumeration, and explaining away
*✍️ By hand · ★★☆*

Use the network from the setup: $P(R=1) = 0.2$, $P(S=1) = 0.4$, $P(W=1\mid R,S)$ = 0.05, 0.80, 0.90, 0.99 for $(R,S) = (0,0),(0,1),(1,0),(1,1)$.

(a) $P(R=1, S=0, W=1)$. (b) $P(W=1)$. (c) $P(R=1\mid W=1)$. (d) $P(R=1\mid W=1, S=1)$. Compare (c) and (d).

In [ ]:
p_joint_a = None
p_w1 = None
p_r_given_w = None
p_r_given_ws = None

_J = np.zeros((2, 2, 2))
for r_, s_, w_ in itertools.product((0, 1), repeat=3):
    _pw = P_W1_given_RS[r_, s_]
    _J[r_, s_, w_] = P_R[r_] * P_S[s_] * (_pw if w_ else 1 - _pw)
check('P(R=1,S=0,W=1)', p_joint_a, _J[1, 0, 1], tol=1e-4)
check('P(W=1)', p_w1, _J[:, :, 1].sum(), tol=1e-4)
check('P(R=1|W=1)', p_r_given_w, _J[1, :, 1].sum() / _J[:, :, 1].sum(), tol=1e-3)
check('P(R=1|W=1,S=1)', p_r_given_ws, _J[1, 1, 1] / _J[:, 1, 1].sum(), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Restriction: keep only $W=1$. Combination: multiply $P(R)P(S)P(W=1\mid R,S)$. Marginalisation: sum out $S$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$P(R=1, W=1) = 0.2(0.6\cdot0.9 + 0.4\cdot0.99) = 0.1872$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $0.2\cdot0.6\cdot0.9 = \mathbf{0.108}$.
(b) $P(R=1,W=1) = 0.1872$; $P(R=0,W=1) = 0.8(0.6\cdot0.05 + 0.4\cdot0.8) = 0.28$; $P(W=1) = \mathbf{0.4672}$.
(c) $0.1872/0.4672 = \mathbf{0.4007}$ (up from the prior 0.2: wet grass is evidence for rain).
(d) $\frac{0.2\cdot0.4\cdot0.99}{0.2\cdot0.4\cdot0.99 + 0.8\cdot0.4\cdot0.8} = \frac{0.0792}{0.0792 + 0.256} = \mathbf{0.2363}$.
Learning that the sprinkler was on **explains away** the wet grass, so rain drops back towards its prior.

```python
p_joint_a = 0.108
p_w1 = 0.4672
p_r_given_w = 0.1872 / 0.4672
p_r_given_ws = 0.0792 / (0.0792 + 0.256)
```

</details>

### Exercise 10 · Beta-Bernoulli updating
*✍️ By hand · ★☆☆*

Prior $\theta\sim\text{Beta}(2, 2)$; we observe 7 heads and 3 tails. Give the posterior parameters, the MLE, the MAP estimate and the posterior predictive
$P(X_{11} = 1\mid\mathcal D)$.

In [ ]:
a_post = None
b_post = None
theta_mle = None
theta_map = None
p_pred = None

check('a_post', a_post, 2 + 7); check('b_post', b_post, 2 + 3)
check('MLE', theta_mle, 7 / 10)
from scipy.optimize import minimize_scalar
check('MAP (mode of posterior)', theta_map, minimize_scalar(lambda t: -beta(9, 5).logpdf(t), bounds=(0.01, 0.99), method='bounded').x, tol=1e-4)
check('predictive = posterior mean', p_pred, beta(9, 5).mean())

<details>
<summary><b>💡 Hint</b></summary>

$\text{Beta}(a+N_1, b+N_0)$; mode $\frac{\alpha-1}{\alpha+\beta-2}$; predictive = posterior mean $\frac{\alpha}{\alpha+\beta}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Posterior $\text{Beta}(9, 5)$. MLE $= 0.7$. MAP $= 8/12 = \mathbf{0.667}$. Predictive $= 9/14\approx\mathbf{0.643}$.
The prior acts like 2 extra heads and 2 extra tails, pulling the estimates towards 0.5.

```python
a_post, b_post = 9, 5
theta_mle = 0.7
theta_map = 8 / 12
p_pred = 9 / 14
```

</details>

### Exercise 11 · Laplace smoothing a CPT
*✍️ By hand · ★☆☆*

Among $N_{\text{draw}} = 3$ training rows with class `draw`, feature `MH` took the values `1a`, `2a`, `3a` with counts $0, 1, 2$.
Compute $P(\text{MH}\mid\text{draw})$ by MLE and with Laplace smoothing $\alpha = 1$. What happens to the smoothed estimates as the counts grow (e.g. $0, 100, 200$)?

In [ ]:
p_mle = None       # [P(1a|draw), P(2a|draw), P(3a|draw)]
p_laplace = None

_c = np.array([0, 1, 2.])
check('MLE', p_mle, _c / _c.sum())
check('Laplace', p_laplace, (_c + 1) / (_c + 1).sum())

<details>
<summary><b>💡 Hint</b></summary>

Denominator with smoothing: $\sum_{k'}(N_{k'} + 1) = 3 + 3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MLE: $(0, \tfrac13, \tfrac23)$. Laplace: $(\tfrac16, \tfrac26, \tfrac36)$. With counts $(0, 100, 200)$: $(\tfrac{1}{303}, \tfrac{101}{303}, \tfrac{201}{303})\approx(0.003, 0.333, 0.663)$:
the smoothing matters only when data is scarce.

```python
p_mle = np.array([0, 1, 2]) / 3
p_laplace = np.array([1, 2, 3]) / 6
```

</details>

### Exercise 12 · One EM iteration by hand
*✍️ By hand · ★★★*

Network $A\to B$ (binary). Data (`A_obs`, `B_obs` in the setup): complete rows $(A,B)$ = (1,1)×3, (1,0)×1, (0,0)×2, (0,1)×1, and three rows with $A$ missing:
$(?,1)$×2 and $(?,0)$×1. Start with $\theta^0$: $P(A=1) = 0.6$, $P(B=1\mid A=1) = 0.8$, $P(B=1\mid A=0) = 0.3$.

**E-step:** compute $P(A=1\mid B=1,\theta^0)$ and $P(A=1\mid B=0,\theta^0)$. **M-step:** compute the expected counts and the new $\theta^1$.

In [ ]:
theta1 = None   # [P(A=1), P(B=1|A=1), P(B=1|A=0)]

# reference: enumerate the 2^3 completions of the missing values, weight each by its posterior probability under theta0
_pa, _p1, _p0 = 0.6, 0.8, 0.3
_miss = np.where(np.isnan(A_obs))[0]
_num = np.zeros(3); _den = np.zeros(3)
for _comp in itertools.product((0, 1), repeat=len(_miss)):
    _A = A_obs.copy(); _A[_miss] = _comp
    _w = np.prod([(_pa if a else 1 - _pa) * ((_p1 if a else _p0) if b else 1 - (_p1 if a else _p0)) for a, b in zip(_A[_miss], B_obs[_miss])])
    _num += _w * np.array([_A.sum(), (_A * B_obs).sum(), ((1 - _A) * B_obs).sum()])
    _den += _w * np.array([len(_A), _A.sum(), (1 - _A).sum()])
check('theta1', theta1, _num / _den, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$P(A=1\mid B=1) = \frac{0.6\cdot0.8}{0.6\cdot0.8 + 0.4\cdot0.3} = 0.8$; $P(A=1\mid B=0) = \frac{0.6\cdot0.2}{0.6\cdot0.2+0.4\cdot0.7} = 0.3$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Expected $N(A=1) = 4 + 2\cdot0.8 + 0.3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

E-step: $P(A=1\mid B=1) = 0.48/0.60 = 0.8$, $P(A=1\mid B=0) = 0.12/0.40 = 0.3$.

Expected counts: $N(A=1) = 4 + 1.6 + 0.3 = 5.9$, $N(A=0) = 3 + 0.4 + 0.7 = 4.1$, $N(A=1,B=1) = 3 + 1.6 = 4.6$, $N(A=0,B=1) = 1 + 0.4 = 1.4$.

M-step: $P(A=1) = 5.9/10 = \mathbf{0.59}$, $P(B=1\mid A=1) = 4.6/5.9 = \mathbf{0.7797}$, $P(B=1\mid A=0) = 1.4/4.1 = \mathbf{0.3415}$.
(The check reproduces this by enumerating all $2^3$ completions of the missing values.)

```python
theta1 = [0.59, 4.6 / 5.9, 1.4 / 4.1]
```

</details>

### Exercise 13 · The posterior predictive integral
*✍️ By hand · ★★☆*

Show that $\int_0^1\theta\,\text{Beta}(\theta\mid\alpha,\beta)\,d\theta = \frac{\alpha}{\alpha+\beta}$, using $\text{Beta}(\theta\mid\alpha,\beta) = \frac{\theta^{\alpha-1}(1-\theta)^{\beta-1}}{B(\alpha,\beta)}$,
$B(\alpha,\beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}$ and $\Gamma(z+1) = z\Gamma(z)$. Conclude $P(X_{N+1} = 1\mid\mathcal D) = \frac{a+N_1}{a+b+N_1+N_0}$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Multiplying by $\theta$ turns the integrand into an unnormalised $\text{Beta}(\alpha+1,\beta)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\int_0^1\theta\frac{\theta^{\alpha-1}(1-\theta)^{\beta-1}}{B(\alpha,\beta)}d\theta = \frac{B(\alpha+1,\beta)}{B(\alpha,\beta)} = \frac{\Gamma(\alpha+1)\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\alpha+\beta+1)} = \frac{\alpha}{\alpha+\beta}$.
With the posterior $\alpha = a+N_1$, $\beta = b+N_0$ this is the predictive formula of the note. Exercise 19 confirms it by Monte Carlo.

</details>

## Part C · Code from scratch

### Exercise 14 · Gaussian MLE and the log-likelihood surface
*💻 Code · ★☆☆*

Implement `gauss_loglik(mu, sigma2, x)` from the formula in the note and `gauss_mle(x)` returning $(\hat\mu, \hat\sigma)$. Verify against `scipy.stats.norm.fit` and
check numerically that the log-likelihood is lower when you move away from the MLE. Optional: contour-plot the surface over $(\mu,\sigma)$.

In [ ]:
def gauss_loglik(mu, sigma2, x):
    return None

def gauss_mle(x):
    return None, None

mle_mu, mle_sigma = gauss_mle(data_gauss)

check('MLE = norm.fit', None if mle_mu is None else [mle_mu, mle_sigma], list(norm.fit(data_gauss)))
if gauss_loglik(0, 1, data_gauss) is not None:
    check('loglik = sum of norm.logpdf', gauss_loglik(1.0, 2.0, data_gauss), norm.logpdf(data_gauss, 1.0, np.sqrt(2.0)).sum())
    _best = gauss_loglik(data_gauss.mean(), data_gauss.var(), data_gauss)
    check('MLE beats perturbed parameters', all(gauss_loglik(data_gauss.mean() + dm, data_gauss.var() * dv, data_gauss) < _best
                                                for dm, dv in [(0.3, 1), (-0.3, 1), (0, 1.3), (0, 0.7)]), True)
else:
    check('loglik = sum of norm.logpdf', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

$\ln L = -\frac{1}{2\sigma^2}\sum(x_i-\mu)^2 - \frac N2\ln(2\pi\sigma^2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`norm.fit` is exactly the closed form ($\hat\sigma$ with $1/N$). The surface is concave in $\mu$ and peaks at the MLE.

```python
def gauss_loglik(mu, sigma2, x):
    return -np.sum((x - mu) ** 2) / (2 * sigma2) - len(x) / 2 * np.log(2 * np.pi * sigma2)

def gauss_mle(x):
    mu = x.mean()
    return mu, np.sqrt(np.mean((x - mu) ** 2))

mle_mu, mle_sigma = gauss_mle(data_gauss)
M, S = np.meshgrid(np.linspace(0, 4.5, 80), np.linspace(1, 4, 80))
LL = np.vectorize(lambda m, s: gauss_loglik(m, s ** 2, data_gauss))(M, S)
plt.figure(figsize=(4.5, 3.5)); plt.contour(M, S, LL, 25); plt.plot(mle_mu, mle_sigma, 'r*', ms=12)
plt.xlabel('mu'); plt.ylabel('sigma'); plt.show()
```

</details>

### Exercise 15 · Learn CPTs by counting
*💻 Code · ★★☆*

Sample 5000 complete cases from the Rain/Sprinkler/Wet network (ancestral sampling: parents before children) into a DataFrame `df_rsw` with columns `R, S, W`.
Then estimate the CPT $\hat P(W=1\mid R, S)$ by counting (a $2\times2$ array `cpt_W_hat`) and $\hat P(R=1)$ (`p_R_hat`).

In [ ]:
def sample_rsw(n, seed=0):
    # TODO: return a DataFrame with integer columns R, S, W
    return None

df_rsw = sample_rsw(5000)
cpt_W_hat = None
p_R_hat = None

if df_rsw is not None and cpt_W_hat is not None:
    check('counts match pandas groupby', cpt_W_hat, df_rsw.groupby(['R', 'S'])['W'].mean().unstack().values)
    check('close to the true CPT', np.abs(np.asarray(cpt_W_hat) - P_W1_given_RS).max() < 0.05, True)
    check('P(R=1) close to 0.2', abs(p_R_hat - 0.2) < 0.03, True)
else:
    check('counts match pandas groupby', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Sample $R$ and $S$ independently, then $W\sim\text{Bernoulli}(P(W=1\mid R,S))$ using `P_W1_given_RS[R, S]` with integer arrays.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\hat\theta_{ijk} = N(X_i = k, pa = j)/N(pa = j)$: for each $(r,s)$ take the fraction of rows with $W=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With complete data the likelihood decomposes per node, and each CPT entry is just a conditional frequency. The rarest parent configuration $(R=1,S=1)$ has only about
$5000\cdot0.08 = 400$ rows, so it is estimated least precisely; with small data such cells can be empty, which is the zero-count problem again.

```python
def sample_rsw(n, seed=0):
    r = np.random.default_rng(seed)
    R = (r.random(n) < P_R[1]).astype(int)
    S = (r.random(n) < P_S[1]).astype(int)
    W = (r.random(n) < P_W1_given_RS[R, S]).astype(int)
    return pd.DataFrame({'R': R, 'S': S, 'W': W})

df_rsw = sample_rsw(5000)
cpt_W_hat = np.array([[df_rsw.W[(df_rsw.R == r_) & (df_rsw.S == s_)].mean() for s_ in (0, 1)] for r_ in (0, 1)])
p_R_hat = df_rsw.R.mean()
print(cpt_W_hat.round(3), round(p_R_hat, 3))
```

</details>

### Exercise 16 · Naive Bayes from frequency tables, with Laplace smoothing
*💻 Code · ★★☆*

The 20-row DataFrame `poker` below mimics the self-study's poker data: class `BH` $\in$ {op, me, draw}, features `MH`, `FC`, `SC`. Note that `draw` never occurs with `MH = '1a'`.
Implement `nb_posterior(df, query, alpha)` returning the posterior over `classes` for a query dict, with CPTs
$\frac{N(X_i = k, BH = c) + \alpha}{N(BH = c) + \alpha\,|\text{values of }X_i|}$ (unsmoothed class priors). Compute the posterior for `query` with $\alpha = 0$ (`post_mle`) and $\alpha = 1$ (`post_laplace`).

In [ ]:
poker = pd.DataFrame([   # BH, MH, FC, SC   (20 rows; 'draw' never occurs with MH = '1a')
    ('op', '1a', 1, 1), ('op', '1a', 2, 0), ('op', '2a', 1, 2), ('op', '2v', 0, 1), ('op', '1a', 1, 0),
    ('op', '3v', 2, 1), ('op', '2a', 0, 0), ('op', '1a', 3, 1), ('op', '2v', 1, 2),
    ('me', '3v', 1, 1), ('me', '2v', 0, 0), ('me', '1a', 2, 2), ('me', '3v', 1, 0), ('me', '2a', 1, 1),
    ('me', '3v', 3, 2), ('me', '2v', 2, 1), ('me', '2a', 0, 1),
    ('draw', '2a', 1, 1), ('draw', '2v', 2, 1), ('draw', '3v', 1, 0)], columns=['BH', 'MH', 'FC', 'SC'])
classes = ['op', 'me', 'draw']
query = {'MH': '1a', 'FC': 1, 'SC': 1}

def nb_posterior(df, query, alpha=0.0):
    # TODO: return np.array of P(BH=c | query) for c in classes
    return None

post_mle = nb_posterior(poker, query, 0.0)
post_laplace = nb_posterior(poker, query, 1.0)

from sklearn.preprocessing import OrdinalEncoder
_enc = OrdinalEncoder().fit(poker[['MH', 'FC', 'SC']])
_cnb = CategoricalNB(alpha=1.0).fit(_enc.transform(poker[['MH', 'FC', 'SC']]), poker.BH)
_p = _cnb.predict_proba(_enc.transform(pd.DataFrame([query])))[0]
check('Laplace posterior = sklearn CategoricalNB(alpha=1)', post_laplace, [_p[list(_cnb.classes_).index(c)] for c in classes])
check('MLE gives draw exactly 0', None if post_mle is None else post_mle[2], 0.0)

<details>
<summary><b>💡 Hint 1</b></summary>

Prior: `np.mean(df.BH == c)`. For each feature: count rows of class `c` with the query value; the number of possible values is `df[col].nunique()`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Multiply prior and the three likelihoods, then normalise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Without smoothing `draw` gets posterior exactly 0, whatever `FC` and `SC` say. With $\alpha = 1$ it gets a small but non-zero probability, and the result
matches scikit-learn's `CategoricalNB`, which implements exactly this smoothed counting.

```python
poker = pd.DataFrame([   # BH, MH, FC, SC   (20 rows; 'draw' never occurs with MH = '1a')
    ('op', '1a', 1, 1), ('op', '1a', 2, 0), ('op', '2a', 1, 2), ('op', '2v', 0, 1), ('op', '1a', 1, 0),
    ('op', '3v', 2, 1), ('op', '2a', 0, 0), ('op', '1a', 3, 1), ('op', '2v', 1, 2),
    ('me', '3v', 1, 1), ('me', '2v', 0, 0), ('me', '1a', 2, 2), ('me', '3v', 1, 0), ('me', '2a', 1, 1),
    ('me', '3v', 3, 2), ('me', '2v', 2, 1), ('me', '2a', 0, 1),
    ('draw', '2a', 1, 1), ('draw', '2v', 2, 1), ('draw', '3v', 1, 0)], columns=['BH', 'MH', 'FC', 'SC'])
classes = ['op', 'me', 'draw']
query = {'MH': '1a', 'FC': 1, 'SC': 1}

def nb_posterior(df, query, alpha=0.0):
    scores = []
    for c in classes:
        dc = df[df.BH == c]
        s = np.mean(df.BH == c)
        for col, val in query.items():
            s *= (np.sum(dc[col] == val) + alpha) / (len(dc) + alpha * df[col].nunique())
        scores.append(s)
    scores = np.array(scores)
    return scores / scores.sum()

post_mle = nb_posterior(poker, query, 0.0)
post_laplace = nb_posterior(poker, query, 1.0)
print('MLE    ', post_mle.round(3)); print('Laplace', post_laplace.round(3))
```

</details>

### Exercise 17 · Restriction, combination, marginalisation in NumPy
*💻 Code · ★★☆*

Write `joint_rsw()` returning the full joint as an array `J[r, s, w]` (combination via broadcasting), and `query(J, target_axis, evidence)` that
**restricts** to the evidence (a dict `{axis: value}`), **marginalises** all other axes and normalises. Reproduce Exercise 9 (c) and (d) and also compute $P(S=1\mid W=1)$.

In [ ]:
AX = {'R': 0, 'S': 1, 'W': 2}

def joint_rsw():
    return None

def query(J, target_axis, evidence):
    # returns the distribution over the target variable (length-2 array)
    return None

J_rsw = joint_rsw()
p_R_given_W = None        # P(R=1 | W=1)
p_R_given_WS = None       # P(R=1 | W=1, S=1)
p_S_given_W = None        # P(S=1 | W=1)

check('P(R=1|W=1)', p_R_given_W, 0.1872 / 0.4672, tol=1e-6)
check('P(R=1|W=1,S=1)', p_R_given_WS, 0.0792 / 0.3352, tol=1e-6)
# P(S=1|W=1) by brute-force loops
_num = sum(P_R[r_] * P_S[1] * P_W1_given_RS[r_, 1] for r_ in (0, 1)); _den = sum(P_R[r_] * P_S[s_] * P_W1_given_RS[r_, s_] for r_ in (0, 1) for s_ in (0, 1))
check('P(S=1|W=1)', p_S_given_W, _num / _den)

<details>
<summary><b>💡 Hint</b></summary>

`P_W = np.stack([1 - P_W1_given_RS, P_W1_given_RS], axis=-1)`; then `J = P_R[:, None, None] * P_S[None, :, None] * P_W`. For restriction use `np.take(J, [v], axis=a)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Restriction = indexing, combination = broadcasting products, marginalisation = `sum` over axes. Enumeration costs $O(2^n)$, fine here, hopeless for large networks
(which is why variable elimination and approximate inference exist, see Lecture 7). $P(S=1\mid W=1)\approx0.7175$: wet grass also raises belief in the sprinkler.

```python
AX = {'R': 0, 'S': 1, 'W': 2}

def joint_rsw():
    P_W = np.stack([1 - P_W1_given_RS, P_W1_given_RS], axis=-1)      # [r, s, w]
    return P_R[:, None, None] * P_S[None, :, None] * P_W

def query(J, target_axis, evidence):
    T = J
    for ax, v in evidence.items():                                   # restriction (keep the axis, length 1)
        T = np.take(T, [v], axis=ax)
    other = tuple(a for a in range(T.ndim) if a != target_axis)      # marginalisation
    m = T.sum(axis=other)
    return m / m.sum()

J_rsw = joint_rsw()
p_R_given_W = query(J_rsw, AX['R'], {AX['W']: 1})[1]
p_R_given_WS = query(J_rsw, AX['R'], {AX['W']: 1, AX['S']: 1})[1]
p_S_given_W = query(J_rsw, AX['S'], {AX['W']: 1})[1]
print(round(p_R_given_W, 4), round(p_R_given_WS, 4), round(p_S_given_W, 4))
```

</details>

### Exercise 18 · EM for a network with a partly hidden variable
*💻 Code · ★★★*

Implement EM for $A\to B$ with the data `A_obs, B_obs` (np.nan = missing). Each iteration: **E-step** soft-assign $w_n = P(A=1\mid b_n,\theta)$ for missing rows
($w_n = a_n$ for observed rows), **M-step** $P(A=1) = \frac1N\sum w_n$, $P(B=1\mid A=1) = \frac{\sum w_nb_n}{\sum w_n}$, $P(B=1\mid A=0) = \frac{\sum(1-w_n)b_n}{\sum(1-w_n)}$.
Also compute the observed-data log-likelihood $\ell(\theta) = \sum_n\ln P(\text{observed part of row }n\mid\theta)$ before every iteration.

Run 100 iterations from $\theta^0 = (0.6, 0.8, 0.3)$. Store the first update in `theta_em_1`, the final parameters in `theta_em` and the list of log-likelihoods in `ll_trace`.

In [ ]:
def observed_loglik(theta, A, B):
    return None

def em_step(theta, A, B):
    return None

theta_em_1 = None
theta_em = None
ll_trace = None

if ll_trace is not None:
    check('first iteration = Exercise 12', theta_em_1, [0.59, 4.6 / 5.9, 1.4 / 4.1], tol=1e-6)
    check('log-likelihood never decreases', bool(np.all(np.diff(ll_trace) >= -1e-10)), True)
    def _nll(z):
        pa, p1, p0 = expit(z); ll = 0
        for a, b in zip(A_obs, B_obs):
            j = lambda a_: (pa if a_ else 1 - pa) * ((p1 if a_ else p0) if b else 1 - (p1 if a_ else p0))
            ll += np.log(j(0) + j(1) if np.isnan(a) else j(int(a)))
        return -ll
    check('EM reaches the maximum of the observed-data likelihood', ll_trace[-1], -minimize(_nll, np.zeros(3)).fun, tol=1e-5)
else:
    check('first iteration = Exercise 12', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

$P(A=1\mid b) = \frac{P(A=1)P(b\mid A=1)}{P(A=1)P(b\mid A=1) + P(A=0)P(b\mid A=0)}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For a missing row $\ln P(b_n) = \ln\sum_aP(a)P(b_n\mid a)$; for a complete row $\ln P(a_n)P(b_n\mid a_n)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The first iteration reproduces the hand computation $(0.59, 0.7797, 0.3415)$, the log-likelihood increases monotonically, and EM converges to about
$(0.583, 0.771, 0.360)$, the same likelihood value a generic optimiser finds by maximising $\ell(\theta)$ directly (the check). EM needs no step size and keeps
the parameters valid probabilities automatically.

```python
def _pb(theta, a, b):
    pa, p1, p0 = theta
    pb1 = p1 if a == 1 else p0
    return (pa if a == 1 else 1 - pa) * (pb1 if b == 1 else 1 - pb1)

def observed_loglik(theta, A, B):
    return sum(np.log(_pb(theta, 0, b) + _pb(theta, 1, b)) if np.isnan(a) else np.log(_pb(theta, int(a), b))
               for a, b in zip(A, B))

def em_step(theta, A, B):
    w = np.array([_pb(theta, 1, b) / (_pb(theta, 0, b) + _pb(theta, 1, b)) if np.isnan(a) else a for a, b in zip(A, B)])
    return np.array([w.mean(), (w * B).sum() / w.sum(), ((1 - w) * B).sum() / (1 - w).sum()])

theta = np.array([0.6, 0.8, 0.3]); ll_trace = []
for it in range(100):
    ll_trace.append(observed_loglik(theta, A_obs, B_obs))
    theta = em_step(theta, A_obs, B_obs)
    if it == 0:
        theta_em_1 = theta.copy()
ll_trace.append(observed_loglik(theta, A_obs, B_obs))
theta_em = theta
print('theta after 1 step', theta_em_1.round(4), ' final', theta_em.round(4), ' ll', round(ll_trace[-1], 4))
```

</details>

### Exercise 19 · Sequential Bayesian updating and a Monte Carlo predictive
*💻 Code · ★★☆*

Simulate 50 flips of a coin with $\theta_{\text{true}} = 0.3$ (use `rng`). Starting from a $\text{Beta}(1,1)$ prior, update **one flip at a time** and record the posterior
mean after every flip (`post_means`, length 50) and the final $(a, b)$ in `ab_final`. Then estimate the posterior predictive
$\int\theta\,p(\theta\mid\mathcal D)\,d\theta$ by sampling 100 000 values of $\theta$ from the posterior (`pred_mc`). Plot the posterior after 0, 5, 20 and 50 flips.

In [ ]:
flips = (rng.random(50) < 0.3).astype(int)
post_means = None
ab_final = None
pred_mc = None

if ab_final is not None:
    check('final posterior = Beta(1+heads, 1+tails)', ab_final, [1 + flips.sum(), 1 + 50 - flips.sum()])
    check('sequential = batch update', post_means[-1], beta(1 + flips.sum(), 1 + 50 - flips.sum()).mean())
    check('Monte Carlo predictive ~ (a)/(a+b)', pred_mc, beta(*ab_final).mean(), tol=5e-3)
else:
    check('final posterior = Beta(1+heads, 1+tails)', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

Each flip: `a += x; b += 1 - x`. `rng.beta(a, b, size=100_000)` samples the posterior.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Updating one observation at a time gives exactly the same posterior as one batch update (conjugacy: yesterday's posterior is today's prior). The posterior narrows like
$1/\sqrt N$ around the true value; the Monte Carlo average matches $\frac{a}{a+b}$ to about three decimals.

```python
flips = (rng.random(50) < 0.3).astype(int)
a, b = 1, 1
post_means = []
snap = {0: (a, b)}
for i, x in enumerate(flips, 1):
    a, b = a + x, b + 1 - x
    post_means.append(a / (a + b))
    if i in (5, 20, 50):
        snap[i] = (a, b)
ab_final = [a, b]
pred_mc = rng.beta(a, b, size=100_000).mean()
th = np.linspace(0, 1, 300)
plt.figure(figsize=(5, 3))
for n_, (aa, bb) in snap.items():
    plt.plot(th, beta(aa, bb).pdf(th), label=f'after {n_} flips')
plt.axvline(0.3, ls='--', c='k'); plt.legend(); plt.show()
```

</details>

### Exercise 20 · EM for a two-component Gaussian mixture
*💻 Code · ★★★*

The hidden variable is now the component label $Z_n\in\{0,1\}$ of each point. Implement EM for a 1-D mixture
$p(x) = \pi\,\mathcal N(x\mid\mu_1,\sigma_1^2) + (1-\pi)\,\mathcal N(x\mid\mu_0,\sigma_0^2)$:
**E-step** responsibilities $r_n = P(Z_n = 1\mid x_n)$, **M-step** weighted MLEs (weighted means/variances, $\pi = \bar r$).
Run 200 iterations from $\mu = (-1, 1)$, $\sigma^2 = (1, 1)$, $\pi = 0.5$, store the log-likelihood trace in `ll_gmm` and compare the final value with
`sklearn.mixture.GaussianMixture`.

In [ ]:
r = np.random.default_rng(0)
x_mix = np.r_[r.normal(-2, 0.8, 150), r.normal(2, 1.2, 250)]

def gmm_em(x, iters=200):
    # TODO: return (params dict, list of log-likelihoods)
    return None, None

gmm_params, ll_gmm = gmm_em(x_mix)

if ll_gmm is not None:
    check('log-likelihood never decreases', bool(np.all(np.diff(ll_gmm) >= -1e-8)), True)
    _gm = GaussianMixture(2, random_state=0, tol=1e-10, max_iter=1000).fit(x_mix[:, None])
    check('same optimum as sklearn', ll_gmm[-1], _gm.score(x_mix[:, None]) * len(x_mix), tol=1e-3)
else:
    check('log-likelihood never decreases', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Log-likelihood: $\sum_n\ln[\pi\mathcal N(x_n\mid\mu_1,\sigma_1^2) + (1-\pi)\mathcal N(x_n\mid\mu_0,\sigma_0^2)]$ with `norm.pdf`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

M-step for component 1: $\mu_1 = \frac{\sum r_nx_n}{\sum r_n}$, $\sigma_1^2 = \frac{\sum r_n(x_n-\mu_1)^2}{\sum r_n}$; component 0 with $1-r_n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is the same E/M structure as Exercise 18 with a continuous child: the E-step computes expected sufficient statistics (soft counts), the M-step plugs them into the
complete-data MLE formulas of Exercise 7. The trace increases monotonically and reaches scikit-learn's optimum (about $-813.7$). Swapping the initial means would give the same solution
with the labels exchanged (label switching).

```python
r = np.random.default_rng(0)
x_mix = np.r_[r.normal(-2, 0.8, 150), r.normal(2, 1.2, 250)]

def gmm_em(x, iters=200):
    mu, var, pi = np.array([-1., 1.]), np.array([1., 1.]), 0.5
    lls = []
    for _ in range(iters + 1):
        p1 = pi * norm.pdf(x, mu[1], np.sqrt(var[1]))
        p0 = (1 - pi) * norm.pdf(x, mu[0], np.sqrt(var[0]))
        lls.append(np.log(p0 + p1).sum())
        rr = p1 / (p0 + p1)
        for k, wk in ((1, rr), (0, 1 - rr)):
            mu[k] = (wk * x).sum() / wk.sum()
            var[k] = (wk * (x - mu[k]) ** 2).sum() / wk.sum()
        pi = rr.mean()
    return {'mu': mu, 'var': var, 'pi': pi}, lls

gmm_params, ll_gmm = gmm_em(x_mix)
print({k: np.round(v, 3) for k, v in gmm_params.items()}, round(ll_gmm[-1], 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*